# YBF DPO Fine-Tuning — Colab Notebook

Trains a small open-source LM with Direct Preference Optimization on YBF-scored preference pairs.
Companion to https://github.com/Guru35/ybf-toy-experiment

**Runtime:** Use **T4 GPU** (Runtime → Change runtime type → T4). Free tier sufficient for 135M model.
**Cost:** $0 (Colab free tier).
**Expected time:** ~20-30 minutes total.

**Methodological note:** Preference pairs derive from the Claude Haiku YBF scorer. Scorer-level fidelity gaps (e.g. GERCEKLIK under-application — see §3.8 of preprint) propagate to the fine-tuned model. This is a methodology demo, not a capacity claim.


## 1. Setup

Install dependencies (~3 min on first run).


In [ ]:
!pip install -q torch transformers datasets peft trl accelerate bitsandbytes

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}, device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')


## 2. Clone the YBF experiment repo

Pulls the DPO dataset and training script from GitHub.


In [ ]:
!git clone -q https://github.com/Guru35/ybf-toy-experiment.git ybf_toy
%cd ybf_toy
!ls data/ybf_dpo_*.jsonl


## 3. Inspect the dataset

Verify the preference pairs loaded correctly.


In [ ]:
import json

with open('data/ybf_dpo_train.jsonl') as f:
    train = [json.loads(l) for l in f]
with open('data/ybf_dpo_test.jsonl') as f:
    test = [json.loads(l) for l in f]
with open('data/ybf_dpo_ood.jsonl') as f:
    ood = [json.loads(l) for l in f]

print(f'Train pairs:    {len(train)}')
print(f'Test pairs:     {len(test)}')
print(f'OOD pairs:      {len(ood)}')

print('\n--- sample train pair ---')
print('PROMPT:  ', train[0]['prompt'][:200], '...')
print('CHOSEN:  ', train[0]['chosen'])
print('REJECTED:', train[0]['rejected'])


## 4. Run DPO training

Trains SmolLM-135M with LoRA + DPO on the YBF preferences.

Override `--model` for larger experiments (TinyLlama-1.1B-Chat, SmolLM-360M).


In [ ]:
# Default: SmolLM-135M, 3 epochs, batch 4
!python ybf_dpo_train.py --epochs 3 --batch_size 4


## 5. Inspect results

Per-set accuracy (chosen > rejected log-probability) and learning curves.


In [ ]:
import json
from pathlib import Path

with open('ybf_dpo_model/eval_results.json') as f:
    eval_results = json.load(f)

for split, r in eval_results.items():
    print(f"{split:>4s}: {r['accuracy_pct']:.1f}% accuracy on {r['n']} pairs "
          f"(mean log-margin: {r['mean_logprob_margin']:+.3f})")

# Loss curve
with open('ybf_dpo_model/training_log.json') as f:
    log = json.load(f)

losses = [(e['step'], e['loss']) for e in log if 'loss' in e]
if losses:
    import matplotlib.pyplot as plt
    steps, vals = zip(*losses)
    plt.figure(figsize=(8, 4))
    plt.plot(steps, vals)
    plt.xlabel('Step'); plt.ylabel('DPO Loss')
    plt.title('YBF DPO Training Loss')
    plt.grid(alpha=0.3)
    plt.show()


## 6. Download the adapter (optional)

Save LoRA weights for later inference.


In [ ]:
from google.colab import files
import shutil

# Zip the adapter and download
shutil.make_archive('ybf_dpo_model_adapter', 'zip', 'ybf_dpo_model/final_adapter')
files.download('ybf_dpo_model_adapter.zip')
files.download('ybf_dpo_model/eval_results.json')


## 7. Quick inference test (optional)

Run the trained model on a sample scenario and inspect its preference.


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import torch

BASE = 'HuggingFaceTB/SmolLM-135M'
tokenizer = AutoTokenizer.from_pretrained(BASE)
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
base_model = AutoModelForCausalLM.from_pretrained(BASE, torch_dtype=torch.bfloat16).cuda()
model = PeftModel.from_pretrained(base_model, 'ybf_dpo_model/final_adapter')
model.eval()

# Pick a trap from OOD
import json
with open('data/ybf_dpo_ood_full.jsonl') as f:
    ood_full = [json.loads(l) for l in f]
ex = ood_full[0]  # first OOD trap (ONUR-decisive)

print('Scenario:')
print(ex['prompt'])
print()
print('Chosen (YBF-aligned, autonomy-respecting):', ex['chosen'])
print('Rejected (paternalist):                   ', ex['rejected'])

def logprob(prompt, response):
    enc_p = tokenizer(prompt, return_tensors='pt').to('cuda')
    enc_f = tokenizer(prompt + response, return_tensors='pt').to('cuda')
    with torch.no_grad():
        logits = model(enc_f.input_ids).logits[:, :-1, :].log_softmax(-1)
    return logits.gather(2, enc_f.input_ids[:, 1:].unsqueeze(-1)).squeeze(-1)[0, enc_p.input_ids.shape[1]-1:].sum().item()

lp_chosen = logprob(ex['prompt'], ex['chosen'])
lp_rejected = logprob(ex['prompt'], ex['rejected'])
print(f'\nModel logprob:')
print(f'  CHOSEN:   {lp_chosen:+.3f}')
print(f'  REJECTED: {lp_rejected:+.3f}')
print(f'  Margin:   {lp_chosen - lp_rejected:+.3f}  {"\u2713 prefers chosen" if lp_chosen > lp_rejected else "\u2717 prefers rejected"}')
